# Task 3 - Consultas Analíticas e Dashboard

Consultas no Amazon Athena sobre o esquema estrela (dados Parquet no S3) e dashboard interativo.

In [ ]:
import awswrangler as wr
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

## 4.1 - Configuração

In [ ]:
GLUE_DATABASE = "classicmodels-etl-db"
ATHENA_WORKGROUP = "classicmodels-etl-workgroup"

## 4.2 - Consulta exploratória em dim_products

In [ ]:
query_products = """
SELECT
    product_id,
    product_name,
    product_line,
    product_vendor
FROM dim_products
LIMIT 20
"""

df_products = wr.athena.read_sql_query(
    sql=query_products,
    database=GLUE_DATABASE,
    workgroup=ATHENA_WORKGROUP
)
df_products

## 4.3 - Vendas totais por país

In [ ]:
query_sales_by_country = """
SELECT
    dim_countries.country,
    SUM(fact_orders.sales_amount) AS total_sales
FROM fact_orders
JOIN dim_countries ON fact_orders.country_key = dim_countries.country_key
GROUP BY dim_countries.country
ORDER BY total_sales DESC
LIMIT 10
"""

df_sales_country = wr.athena.read_sql_query(
    sql=query_sales_by_country,
    database=GLUE_DATABASE,
    workgroup=ATHENA_WORKGROUP
)
df_sales_country

## 4.4 - Detalhamento por data, linha de produto, produto e país

In [ ]:
query_detail = """
SELECT
    dim_dates.full_date,
    dim_products.product_line,
    dim_products.product_name,
    dim_countries.country,
    SUM(fact_orders.sales_amount) AS total_sales
FROM fact_orders
JOIN dim_products ON fact_orders.product_id = dim_products.product_id
JOIN dim_countries ON fact_orders.country_key = dim_countries.country_key
JOIN dim_dates ON fact_orders.order_date_key = dim_dates.date_key
GROUP BY
    dim_dates.full_date,
    dim_products.product_line,
    dim_products.product_name,
    dim_countries.country
"""

df_detail = wr.athena.read_sql_query(
    sql=query_detail,
    database=GLUE_DATABASE,
    workgroup=ATHENA_WORKGROUP
)
df_detail["full_date"] = pd.to_datetime(df_detail["full_date"])
df_detail.head(10)

## 4.5 - Dashboard interativo

In [ ]:
date_min = df_detail["full_date"].min()
date_max = df_detail["full_date"].max()

countries = sorted(df_detail["country"].unique().tolist())
product_lines = sorted(df_detail["product_line"].unique().tolist())

date_start = widgets.DatePicker(description="De:", value=date_min.date())
date_end = widgets.DatePicker(description="Até:", value=date_max.date())

country_select = widgets.SelectMultiple(
    options=["Todos"] + countries,
    value=["Todos"],
    description="País:",
    rows=6
)

product_line_select = widgets.SelectMultiple(
    options=["Todos"] + product_lines,
    value=["Todos"],
    description="Linha:",
    rows=6
)

top_n_slider = widgets.IntSlider(
    value=10, min=1, max=20, step=1,
    description="Top N:"
)

output = widgets.Output()


def update_dashboard(*args):
    with output:
        clear_output(wait=True)

        filtered = df_detail.copy()

        filtered = filtered[
            (filtered["full_date"] >= pd.Timestamp(date_start.value)) &
            (filtered["full_date"] <= pd.Timestamp(date_end.value))
        ]

        if "Todos" not in country_select.value:
            filtered = filtered[filtered["country"].isin(country_select.value)]

        if "Todos" not in product_line_select.value:
            filtered = filtered[filtered["product_line"].isin(product_line_select.value)]

        top_products = (
            filtered
            .groupby("product_name", as_index=False)["total_sales"]
            .sum()
            .nlargest(top_n_slider.value, "total_sales")
            .sort_values("total_sales", ascending=True)
        )

        fig, ax = plt.subplots(figsize=(10, max(4, top_n_slider.value * 0.5)))
        sns.barplot(data=top_products, x="total_sales", y="product_name", ax=ax, palette="viridis")
        ax.set_xlabel("Total Sales ($)")
        ax.set_ylabel("")
        ax.set_title(f"Top {top_n_slider.value} Produtos por Vendas")
        plt.tight_layout()
        plt.show()


date_start.observe(update_dashboard, names="value")
date_end.observe(update_dashboard, names="value")
country_select.observe(update_dashboard, names="value")
product_line_select.observe(update_dashboard, names="value")
top_n_slider.observe(update_dashboard, names="value")

controls = widgets.VBox([
    widgets.HBox([date_start, date_end]),
    widgets.HBox([country_select, product_line_select]),
    top_n_slider
])

display(controls, output)
update_dashboard()